<b>Inventory Planning Problem</b>

In [1]:
# Data

# Multiperiod models rely on an ordered set of time periods.
# Since standard sets in Python are _not_ ordered, we will 
# use an integer range, which is ordered (numbers 1 through 4):
Months = range(1,5)
Demand = {1: 100, 2 : 150, 3 : 150, 4 : 250}
ProductionCost = {1 : 11, 2 : 14, 3 : 15, 4 : 16}
ProductionCapacity = 200
HoldingCost = 2
StorageCapacity = 100

In [2]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [3]:
# variables
Produced = mdl.continuous_var_dict(Months, lb=0, ub=ProductionCapacity, name='Production in month')

In [4]:
# auxiliary variables: inventory at end of month
Inventory = mdl.continuous_var_dict(Months, lb=0, ub=StorageCapacity, name='Inventory at end of month')

In [5]:
# objective: minimize cost
TotalProdCost = mdl.sum(ProductionCost[m]*Produced[m] for m in Months)
TotalHoldingCost = mdl.sum(HoldingCost*Inventory[m] for m in Months)
mdl.minimize(TotalProdCost + TotalHoldingCost)

mdl.add_kpi(TotalProdCost, 'Total production cost')
mdl.add_kpi(TotalHoldingCost, 'Total holding cost')

DecisionKPI(name=Total holding cost,expr=2Inventory at end of month_1+2Inventory at end of month_2+2Inven..)

In [6]:
# inventory balance constraints
for m in Months:
    if m==1:
        mdl.add_constraint(Inventory[1] == Produced[1] - Demand[1])
    else:
        mdl.add_constraint(Inventory[m] == Inventory[m-1] + Produced[m] - Demand[m])

In [7]:
# solve 
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.0103409,status='optimal')

In [8]:
mdl.print_solution()

objective: 9400.000
status: OPTIMAL_SOLUTION(2)
  "Production in month_1"=200.000
  "Production in month_2"=50.000
  "Production in month_3"=200.000
  "Production in month_4"=200.000
  "Inventory at end of month_1"=100.000
  "Inventory at end of month_3"=50.000


In [9]:
mdl.report()

* model docplex_model1 solved with objective = 9400.000
*  KPI: Total production cost = 9100.000
*  KPI: Total holding cost    = 300.000
